# **OOP Project Planning and Design**

**DS212 — Object-Oriented Programming**

**Week 15:** · **Suggested class time: 5 hours (2-hour lecture + 3-hour laboratory)**

**ILO :** Define a real-world problem, identify objects and responsibilities, and create an OOP project design.

The final weeks of DS212 bring everything together in an **object-oriented project**: a tested, packaged Python application that addresses a real problem. Good projects are not started by writing code — they start with a clear **problem**, **requirements**, and a **design**.

```{mermaid}
flowchart LR
    W15["Week 15<br/>Plan & design"] --> W16["Week 16<br/>Implement"]
    W16 --> W17["Week 17<br/>Complete & present"]
    W15 -.- A["proposal<br/>class design"]
    W16 -.- B["working code<br/>tests, Git history"]
    W17 -.- C["package, docs<br/>demo"]
```

Throughout this lesson, we follow one example project: **CampusEats**, a pre-ordering system for a university canteen.

## **Problem Definition**

A project begins with a **problem statement** — a short description of *who* has *what problem*, and *why it matters*.

> **Template:** *[Users]* need a way to *[goal]* because *[current situation / pain]*. Our application will *[what it does]*, so that *[measurable benefit]*.

> **CampusEats:** Students at the main campus need a way to **pre-order meals** because lunch-hour queues at the canteen take 20–30 minutes. Our application will let students browse the menu, place and pay for orders ahead of time, and let canteen staff manage order preparation, **so that** waiting time and food waste are reduced.

### **Stakeholders and scope**

| Stakeholder   | Interest                                                |
| ------------- | ------------------------------------------------------- |
| Students      | order quickly, know when food is ready                  |
| Canteen staff | see incoming orders in order, track preparation         |
| Canteen owner | know which items sell, reduce unsold food               |

Defining **scope** is equally important. Write down what is **in** and **out** of the project:

| In scope (this semester)                 | Out of scope                     |
| ---------------------------------------- | -------------------------------- |
| menu, orders, order status, daily report | real online payment              |
| command-line or notebook interface       | mobile app                       |
| data saved to CSV/JSON files             | multi-campus deployment          |

A small project that **works** is far better than a large one that does not.

## **Requirements**

### **Functional requirements** — what the system must **do**

Writing them as **user stories** keeps the focus on users:

> *As a* **student**, *I want to* **place an order for several menu items**, *so that* **my food is prepared before I arrive.**

Each story needs **acceptance criteria** that can later become **tests**:

| ID   | User story                                           | Acceptance criteria                                                         |
| ---- | ---------------------------------------------------- | --------------------------------------------------------------------------- |
| FR-1 | As a student, I want to view the menu                | only available items are listed, with prices                                 |
| FR-2 | As a student, I want to place an order               | an order has ≥ 1 item; total = Σ price × quantity; unavailable items are rejected |
| FR-3 | As staff, I want to update an order's status         | status moves only forward: *placed → preparing → ready → claimed*            |
| FR-4 | As the owner, I want a daily sales report            | report lists quantity and revenue per item                                   |

### **Non-functional requirements** — **qualities** the system must have

| ID    | Requirement                                                              |
| ----- | ------------------------------------------------------------------------ |
| NFR-1 | invalid input never crashes the program; clear error messages are shown |
| NFR-2 | core classes have unit tests                                             |
| NFR-3 | the project installs with `pip install -e .` and runs on Python ≥ 3.9    |
| NFR-4 | personal data is limited to name and student ID (Data Privacy Act)       |

## **Class-Responsibility Modeling**

### **Step 1 — Candidate classes from the requirements**

Underline the **nouns** (possible classes or attributes) and **verbs** (possible methods):

> A *student* places an *order* for several *menu items*. *Staff* update the *order's status*. The *owner* views a daily *sales report*.

| Candidate     | Decision                                                  |
| ------------- | --------------------------------------------------------- |
| Student       | class — `Customer`                                        |
| Order         | class — `Order`                                           |
| Menu item     | class — `MenuItem`                                        |
| Menu          | class — `Menu` (a collection of items)                    |
| Order line    | class — `OrderLine` (item + quantity) — discovered while modeling totals |
| Status        | an **enumeration** (`OrderStatus`), not a full class       |
| Sales report  | class — `SalesReport` (computed from orders)              |
| Staff, owner  | roles of users of the interface — not needed as classes yet |

### **Step 2 — CRC cards**

| Class          | Responsibilities                                              | Collaborators            |
| -------------- | ------------------------------------------------------------- | ------------------------ |
| `MenuItem`     | know name, price, availability                                | —                        |
| `Menu`         | store items; find items; list available items                 | `MenuItem`               |
| `OrderLine`    | know item and quantity; compute subtotal                      | `MenuItem`               |
| `Order`        | hold lines; compute total; **enforce** valid status changes   | `OrderLine`, `OrderStatus` |
| `Canteen`      | accept orders; queue orders for staff; produce reports        | `Menu`, `Order`, `SalesReport` |
| `SalesReport`  | summarize quantity and revenue per item                       | `Order`                  |

Notice the **information expert** principle: `Order` enforces its **own** status rules, and `OrderLine` computes its **own** subtotal.

## **UML Design**

### **Class diagram**

```{mermaid}
classDiagram
class MenuItem {
    +String code
    +String name
    +float price
    +bool available
}
class Menu {
    -dict items
    +add(item)
    +get(code) MenuItem
    +available() list
}
class OrderLine {
    +MenuItem item
    +int quantity
    +subtotal() float
}
class OrderStatus {
    <<enumeration>>
    PLACED
    PREPARING
    READY
    CLAIMED
}
class Order {
    +int order_id
    +String customer
    -list lines
    -OrderStatus status
    +add_line(item, qty)
    +total() float
    +advance()
}
class Canteen {
    -Menu menu
    -list orders
    +place_order(customer, items) Order
    +queue() list
    +report() SalesReport
}
class SalesReport {
    +dict quantities
    +dict revenue
    +display()
}
Menu o-- MenuItem
Order *-- OrderLine
OrderLine --> MenuItem
Order --> OrderStatus
Canteen *-- Menu
Canteen o-- Order
Canteen ..> SalesReport : creates
```

### **Sequence diagram for the main scenario (FR-2)**

```{mermaid}
sequenceDiagram
    participant S as Student (UI)
    participant C as Canteen
    participant M as Menu
    participant O as Order
    S->>C: place_order("Ana", {"RICE": 1, "ADOBO": 1})
    C->>M: get("ADOBO")
    M-->>C: MenuItem
    C->>O: add_line(item, 1)
    O-->>C: ok
    C-->>S: Order #1, total ₱95.00
```

### **Checking the design against the four core concepts**

| Concept        | Where it appears in CampusEats                                          |
| -------------- | ----------------------------------------------------------------------- |
| Abstraction    | `Canteen.place_order()` hides menu look-ups and line creation          |
| Encapsulation  | `Order` keeps `status` private; only `advance()` may change it          |
| Inheritance    | (optional) `ComboItem(MenuItem)` with a discounted price                |
| Polymorphism   | report exporters (`CSVExporter`, `TextExporter`) share `export()`      |
| Composition    | `Order` is composed of `OrderLine`s; `Canteen` owns a `Menu`            |

Do not force a concept into a design where it does not fit — but if your design uses **none** of them, revisit it.

### **Prototype the riskiest part**

A short **prototype** of the core classes, written before the full project, quickly shows whether the design works:

In [ ]:
from enum import Enum


class OrderStatus(Enum):
    PLACED = 1
    PREPARING = 2
    READY = 3
    CLAIMED = 4


class MenuItem:
    def __init__(self, code, name, price, available=True):
        self.code, self.name, self.price, self.available = code, name, price, available


class OrderLine:
    def __init__(self, item, quantity):
        if quantity < 1:
            raise ValueError("Quantity must be at least 1.")
        self.item, self.quantity = item, quantity

    def subtotal(self):
        return self.item.price * self.quantity


class Order:
    def __init__(self, order_id, customer):
        self.order_id, self.customer = order_id, customer
        self._lines = []
        self._status = OrderStatus.PLACED

    @property
    def status(self):
        return self._status

    def add_line(self, item, quantity):
        if not item.available:
            raise ValueError(f"{item.name} is not available.")
        self._lines.append(OrderLine(item, quantity))

    def total(self):
        return sum(line.subtotal() for line in self._lines)

    def advance(self):
        if self._status is OrderStatus.CLAIMED:
            raise ValueError("Order already claimed.")
        self._status = OrderStatus(self._status.value + 1)


rice = MenuItem("RICE", "Rice", 15)
adobo = MenuItem("ADOBO", "Chicken adobo", 80)

order = Order(1, "Ana")
order.add_line(rice, 1)
order.add_line(adobo, 1)
print(f"Order #{order.order_id} total: ₱{order.total():.2f}, status {order.status.name}")
order.advance()
order.advance()
print("Status now:", order.status.name)

## **Project Structure, Roles, and Milestones**

### **Repository structure**

Use the structure from Week 13 from the very first commit:

```text
campuseats/
├── pyproject.toml
├── README.md
├── LICENSE
├── docs/
│   ├── proposal.md
│   └── class-diagram.md
├── src/campuseats/
│   ├── __init__.py
│   ├── menu.py
│   ├── orders.py
│   ├── canteen.py
│   ├── reports.py
│   └── exceptions.py
├── tests/
│   ├── test_menu.py
│   └── test_orders.py
└── demo.ipynb
```

### **Team roles**

Every member writes code, but each also **owns** an area:

| Role                    | Responsibilities                                                  |
| ----------------------- | ----------------------------------------------------------------- |
| **Project lead**        | coordinates tasks and milestones; keeps the proposal and scope    |
| **Design lead**         | maintains the UML diagrams; reviews class responsibilities        |
| **Quality lead**        | owns the test suite and CI; checks that tests pass before merges  |
| **Documentation lead**  | README, docstrings, demo notebook, presentation materials         |

### **Milestones**

```{mermaid}
gantt
    title CampusEats project plan
    dateFormat YYYY-MM-DD
    section Week 15
    Proposal & requirements      :a1, 2026-11-16, 3d
    CRC cards & UML              :a2, after a1, 3d
    section Week 16
    Core classes + unit tests    :b1, 2026-11-23, 3d
    Interface & persistence      :b2, after b1, 3d
    section Week 17
    Integration & packaging      :c1, 2026-11-30, 3d
    Documentation & rehearsal    :c2, after c1, 2d
    Presentation                 :milestone, c3, 2026-12-05, 0d
```

### **Risks**

List what could go wrong and what you will do about it — for example, *"A member is unavailable for a week → every module has a second reviewer who can continue it."*

## **The Project Proposal**

Your proposal should contain:

1. **Title and team members** with roles.
2. **Problem statement**, stakeholders, and scope (in / out).
3. **Requirements**: user stories with acceptance criteria; non-functional requirements.
4. **Design**: CRC cards, class diagram, and at least one sequence diagram.
5. **Project structure** and planned package layout.
6. **Milestones** and task assignments.
7. **Risks** and how you will handle them.
8. **References** and an **AI-use acknowledgement**, if applicable.

## **Summary**

| Step                     | Output                                                     |
| ------------------------ | ---------------------------------------------------------- |
| Problem definition       | problem statement, stakeholders, scope                     |
| Requirements             | user stories with acceptance criteria; NFRs               |
| Class-responsibility modeling | candidate classes, CRC cards                           |
| UML                      | class diagram, sequence diagram                            |
| Planning                 | repository structure, roles, milestones, risks            |

## **References**

Beck, K., & Cunningham, W. (1989). A laboratory for teaching object-oriented thinking. *ACM SIGPLAN Notices, 24*(10), 1–6. https://doi.org/10.1145/74878.74879

Fowler, M. (2003). *UML distilled: A brief guide to the standard object modeling language* (3rd ed.). Addison-Wesley.

Lott, S. F., & Phillips, D. (2021). *Python object-oriented programming* (4th ed.). Packt Publishing.

Martin, R. C. (2008). *Clean code: A handbook of agile software craftsmanship*. Prentice Hall.

Object Management Group. (2017). *OMG Unified Modeling Language (UML), version 2.5.1*. https://www.omg.org/spec/UML/2.5.1

Python Packaging Authority. (2026). *src layout vs flat layout*. https://packaging.python.org/en/latest/discussions/src-layout-vs-flat-layout/

Republic of the Philippines. (2012). *Republic Act No. 10173: Data Privacy Act of 2012*. National Privacy Commission. https://privacy.gov.ph/data-privacy-act/

Thomas, D., & Hunt, A. (2019). *The pragmatic programmer: Your journey to mastery* (20th anniversary ed.). Addison-Wesley.